In [ ]:
# ---------------------------------------------
# Silver Layer Processing - Hospital dimension
# ---------------------------------------------
# Purpose:
# - Read streaming data from Bronze layer (Delta table)
# - Apply basic data quality rules (deduplication)
# - Mask sensitive information (PII protection using hashing)
# - Add audit column (load timestamp)
# - Perform incremental upsert (MERGE) into Silver Delta table
#
# Key Design:
# - Uses Structured Streaming with foreachBatch for micro-batch processing
# - Implements idempotent upsert logic using Delta MERGE
# - Supports schema evolution and scalable ingestion
# - Checkpointing ensures fault tolerance and exactly-once processing
# ------------------------------------------------------------------------

from delta.tables import DeltaTable
from pyspark.sql.window import Window 
from pyspark.sql.functions import (
    row_number, 
    sha2, 
    col, 
    current_timestamp, 
    monotonically_increasing_id, 
    concat_ws
)
from delta.tables import DeltaTable

silver_hospital_table = "DE_Learning_LH.silver.dim_hospital"

df_hospital_bronze = (
    spark.readStream.table("DE_Learning_LH.hospital_bronze.hospital")
        .dropDuplicates(["hospital_id"])
        .withColumn("load_timestamp", current_timestamp())
)

def merge_dim_hospital(batch_df, batch_id):
    if not spark.catalog.tableExists(silver_hospital_table):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_hospital_table)
        return

    # Load Delta table by name and upsert
    dim_hospital = DeltaTable.forName(spark, silver_hospital_table)

    (
        dim_hospital.alias("t")
            .merge(
                batch_df.alias("s"),
                "t.hospital_id = s.hospital_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
    )

(
    df_hospital_bronze.writeStream
        .foreachBatch(merge_dim_hospital)
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", "Files/healthcare_data/checkpoint/silver_hospital/")
        .start()
)